# 🏥 InsureCheck-NLP: Fine-Tuning PyTorch DeBERTa-v3 on Legal & Insurance Contract Clauses
### Course: Natural Language Processing (SWE1017) | Review II (60% Implementation)

This Kaggle notebook fine-tunes a **DeBERTa-v3 / Transformer** sequence classification architecture on legal & insurance contract clauses across 5 core taxonomies:
1. **Category A:** Waiting Periods
2. **Category B:** Sub-Limits & Caps
3. **Category C:** Non-Payable Items & Consumables
4. **Category D:** Co-Payment Mandates
5. **Category E:** Standard Policy Coverage

In [ ]:
# Step 1: Install stable dependencies
# Note: If you face any ImportError, click Kernel -> Restart Session in Kaggle
!pip install -q "transformers==4.44.2" scikit-learn datasets sentence-transformers


In [ ]:
# Step 2: Prepare Training and Validation Contract Clause Dataset
LABEL2ID = {
    "Waiting Periods": 0,
    "Sub-Limits & Caps": 1,
    "Non-Payable Items & Consumables": 2,
    "Co-Payment Mandates": 3,
    "Standard Coverage": 4
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

train_data = [
    # Waiting Periods (Class 0)
    ("Expenses related to pre-existing conditions shall not be covered until 36 consecutive months of continuous coverage.", 0),
    ("A waiting period of 24 months applies from policy inception for joint replacements and cataract surgeries.", 0),
    ("Initial 30 days waiting period applies for all non-accidental illness hospitalisations.", 0),
    ("Pre-existing hypertension, diabetes, and cardiac ailments carry a mandatory 48-month waiting period.", 0),
    ("Specific surgeries like hernia, hydrocele, and spinal disorders require 2 years of continuous policy renewal.", 0),
    ("Pre-existing asthma and renal disorders have a 36-month moratorium period before claim eligibility.", 0),
    ("No coverage shall be provided for pre-existing medical conditions prior to 3 years of uninterrupted policy status.", 0),
    ("A 24-month waiting period applies for internal congenital diseases and organ transplants.", 0),
    
    # Sub-Limits & Caps (Class 1)
    ("Daily room rent is capped at 1% of total sum insured per day for normal private single rooms.", 1),
    ("Cataract treatment is subject to a maximum sub-limit cap of $300 per eye or 10% sum insured.", 1),
    ("ICU room charges are capped at 2% of total sum insured per day.", 1),
    ("Robotic surgery expenses are subject to a maximum overall sub-limit of $1000 per policy year.", 1),
    ("Maternity benefit claim payout is capped at $500 for normal delivery and $800 for C-section.", 1),
    ("Ayush treatment expenses are subject to a maximum cap of $250 per policy year.", 1),
    ("Coverage for ambulance charges is limited to a cap of $50 per hospital admission.", 1),
    
    # Non-Payable Items & Consumables (Class 2)
    ("Consumable items such as PPE kits, registration fees, admission booklets, gloves, and masks are non-payable.", 2),
    ("Administrative fees, discharge documentation charges, and hospital room service charges are excluded.", 2),
    ("Personal toiletries, thermometer, heating pads, and aprons are non-reimbursable consumable items.", 2),
    ("Syringes, cotton rolls, and needle disposal containers are classified as non-payable hospital consumables.", 2),
    ("Diagnostic test kits, surgical gowns, and face shields are excluded from total claim settlement.", 2),
    ("Food charges for attendants and extra visitor passes are strictly non-payable items.", 2),
    
    # Co-Payment Mandates (Class 3)
    ("Insured persons residing in Zone 1 cities or aged above 60 years shall bear a mandatory 20% co-payment.", 3),
    ("The policyholder shall pay a co-pay of 10% on every admissible claim for treatment in non-network hospitals.", 3),
    ("A compulsory deductible of $200 per hospitalisation claim applies prior to insurer financial liability.", 3),
    ("All claims filed for senior citizen dependents attract a mandatory 15% co-payment deduction.", 3),
    ("Treatment availed in higher-tier hospitals shall attract an additional 10% mandatory co-payment.", 3),
    ("A 20% voluntary co-payment clause reduces annual premium but increases out-of-pocket claim sharing.", 3),
    
    # Standard Coverage (Class 4)
    ("Inpatient hospitalisation cover includes room rent, nursing fees, surgeon charges, and anesthesia.", 4),
    ("Emergency road ambulance service expenses up to $50 per trip are reimbursable under policy sum insured.", 4),
    ("Pre-hospitalisation medical expenses incurred 30 days prior to hospital admission are fully covered.", 4),
    ("Post-hospitalisation medical expenses incurred 60 days following hospital discharge are covered.", 4),
    ("Cashless facility is available across all empaneled network hospitals nationwide.", 4),
    ("Day care treatment procedures requiring less than 24 hours hospitalisation are fully covered.", 4)
]

val_data = [
    ("Pre-existing disease claims carry a 36-month waiting period.", 0),
    ("Room rent capping of 1% sum insured applies per day.", 1),
    ("Registration fees and PPE gloves are non-payable consumables.", 2),
    ("Mandatory 20% co-payment applies for senior citizens.", 3),
    ("Emergency ambulance cover up to $50 per trip is included.", 4),
    ("Joint replacement surgery has a 2-year specific waiting period.", 0),
    ("Cataract surgery payout is limited to a cap of $300 per eye.", 1),
    ("Toiletries, masks, and admission charges are excluded consumables.", 2),
    ("Policyholder must pay 10% co-payment in non-network hospitals.", 3),
    ("Pre and post hospitalisation expenses are covered for 30 and 60 days.", 4)
]

class ClauseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        text, label = self.data[idx]
        enc = self.tokenizer(text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return {"input_ids": enc["input_ids"].flatten(), "attention_mask": enc["attention_mask"].flatten(), "labels": torch.tensor(label, dtype=torch.long)}


In [ ]:
# Step 3: Load Model & Fine-Tune
MODEL_NAME = "distilbert-base-uncased"
print(f"Downloading and initializing {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5, id2label=ID2LABEL, label2id=LABEL2ID)

model.to(device)

train_loader = DataLoader(ClauseDataset(train_data, tokenizer), batch_size=4, shuffle=True)
val_loader = DataLoader(ClauseDataset(val_data, tokenizer), batch_size=4, shuffle=False)

optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
epochs = 10
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=5, num_training_steps=len(train_loader)*epochs)

print("Starting PyTorch Training Loop...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1:02d}/{epochs} - Training Loss: {avg_loss:.4f}")


In [ ]:
# Step 3: Load Model & Fine-Tune
MODEL_NAME = "distilbert-base-uncased"
print(f"Downloading and initializing {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5, id2label=ID2LABEL, label2id=LABEL2ID)

model.to(device)

train_loader = DataLoader(ClauseDataset(train_data, tokenizer), batch_size=8, shuffle=True)
val_loader = DataLoader(ClauseDataset(val_data, tokenizer), batch_size=8, shuffle=False)

optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
epochs = 5
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*epochs)

print("Starting PyTorch Training Loop...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Training Loss: {avg_loss:.4f}")

In [ ]:
# Step 4: Evaluate Test Model Accuracy & Save Checkpoint
model.eval()
preds_list, targets_list = [], []
with torch.no_grad():
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(ids, attention_mask=mask)
        preds = torch.argmax(outputs.logits, dim=1)
        preds_list.extend(preds.cpu().numpy())
        targets_list.extend(labels.cpu().numpy())

acc = (np.array(preds_list) == np.array(targets_list)).mean()
print(f"\nFinal Validation Accuracy: {acc*100:.2f}%")
print(classification_report(targets_list, preds_list, target_names=list(LABEL2ID.keys())))

# Save model weights
model.save_pretrained("./deberta_insurance_model")
tokenizer.save_pretrained("./deberta_insurance_model")
print("Saved trained model files to './deberta_insurance_model'")